# Day 2 FOXF1 / BMP4 reporters — 00_manifest_qc

**Feeds:** Fig 2g, through notebook `03`

**Position in the chain:** run `00`, `01`, `02`, `03` in order, from this lane's directory (`imaging/foxf1_bmp4_day2_fig2g/`).

Ported from the original analysis `FOXF1_BMP4_day2_expression/notebooks/00_manifest_qc.ipynb`.

**Changes from the original notebook**
1. No code cell of the original was edited.
2. The notebook ships without outputs, as the original notebook file does.
3. CZI files are read through `src/trunk_morph_ref/czi_compat.py` instead of `czifile`, a change of one import in `scripts/day2_quantification_helpers.py`.


            # 00 | Manifest And Metadata QC

            ## Notebook Scope

            This notebook inventories the canonical raw inputs, confirms channel metadata,
            and gives an image-first overview of the collaborator-selected review plane for each stack.
            

In [ ]:
import os
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent.resolve()
else:
    ROOT = CWD

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Python:", sys.executable)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scripts import build_analysis_manifest as bam
from scripts import day2_quantification_helpers as dqh

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

            ## Settings Explained

            - Canonical raw inputs are the `.czi` files in `data/with DAPI/`.
            - Paired `.tif` files are treated as review-plane metadata only.
            - The locked biological mapping is:
              `TL Brightfield -> brightfield`, `DAPI -> dapi`, `mCherry -> FOXF1-RFP`, `TagYFP -> BMP4-YFP`.
            

In [ ]:
DATA_DIR = ROOT / "data" / "with DAPI"
MANIFEST_OUTPUT = ROOT / "results" / "manifests" / "raw_input_manifest.tsv"
WRITE_OUTPUTS = True

print("DATA_DIR:", DATA_DIR)
print("MANIFEST_OUTPUT:", MANIFEST_OUTPUT)
print("WRITE_OUTPUTS:", WRITE_OUTPUTS)

In [ ]:
manifest_res = bam.run_manifest_pipeline(
    root=ROOT,
    data_dir=DATA_DIR,
    output=MANIFEST_OUTPUT,
    write_output=WRITE_OUTPUTS,
)

manifest_df = manifest_res["manifest_df"].copy().sort_values("file_id").reset_index(drop=True)
print(manifest_res["summary_text"])
display(manifest_df)

In [ ]:
format_summary = (
    manifest_df.groupby(["source_format", "size_z", "size_y", "size_x"], as_index=False)
    .size()
    .sort_values(["source_format", "size_z", "size_y", "size_x"])
)
channel_summary = manifest_df[["file_name", "raw_channel_names", "canonical_channel_names"]].copy()
review_summary = manifest_df[
    [
        "file_name",
        "paired_review_tif_name",
        "selected_review_z_1based",
        "size_z",
        "paired_tif_axes",
        "paired_tif_size_z",
        "paired_tif_size_y",
        "paired_tif_size_x",
        "pixel_size_x_um",
        "pixel_size_y_um",
        "pixel_size_z_um",
    ]
].copy()

display(format_summary)
display(channel_summary)
display(review_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

z_depth_counts = manifest_df["size_z"].value_counts().sort_index()
axes[0].bar(z_depth_counts.index.astype(str), z_depth_counts.values, color="#546e7a")
axes[0].set_title("Canonical stack depth")
axes[0].set_xlabel("z planes per stack")
axes[0].set_ylabel("stack count")

review_z_counts = (
    manifest_df["selected_review_z_1based"]
    .fillna(-1)
    .astype(int)
    .value_counts()
    .sort_index()
)
review_z_counts = review_z_counts[review_z_counts.index > 0]
axes[1].bar(review_z_counts.index.astype(str), review_z_counts.values, color="#00897b")
axes[1].set_title("Collaborator-selected review z")
axes[1].set_xlabel("review z plane (1-based)")
axes[1].set_ylabel("stack count")

plt.show()

            ## Selected Review-Plane Overview

            This is the quickest image-first sanity check for the whole dataset:
            brightfield context, DAPI tissue pattern, FOXF1, BMP4, and a simple FOXF1/BMP4 composite
            from the collaborator-selected review plane in each stack.
            

In [ ]:
def _robust_rescale(image: np.ndarray, q_low: float = 0.01, q_high: float = 0.995) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr, dtype=np.float32)
    lo = float(np.quantile(finite, q_low))
    hi = float(np.quantile(finite, q_high))
    if not np.isfinite(lo):
        lo = 0.0
    if not np.isfinite(hi) or hi <= lo:
        hi = lo + 1.0
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def plot_selected_review_plane_montage() -> None:
    if manifest_df.empty:
        print("Manifest is empty.")
        return

    fig, axes = plt.subplots(len(manifest_df), 5, figsize=(18, 3.4 * len(manifest_df)), constrained_layout=True)
    if len(manifest_df) == 1:
        axes = np.asarray([axes])

    for ax_row, row in zip(axes, manifest_df.itertuples(index=False)):
        stack = dqh.load_czi_stack(ROOT / row.file_path)
        z_index = int(row.selected_review_z_0based) if pd.notna(row.selected_review_z_0based) else int(stack.data_czyx.shape[1] // 2)

        bf = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "brightfield"), z_index]
        dapi = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "dapi"), z_index]
        foxf1 = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "foxf1"), z_index]
        bmp4 = stack.data_czyx[dqh.channel_index(stack.canonical_channel_names, "bmp4"), z_index]
        composite = np.dstack([
            _robust_rescale(foxf1),
            _robust_rescale(bmp4),
            np.zeros_like(foxf1, dtype=np.float32),
        ])

        panels = [
            ("BF", _robust_rescale(bf), "gray"),
            ("DAPI", _robust_rescale(dapi), "gray"),
            ("FOXF1", _robust_rescale(foxf1), "magma"),
            ("BMP4", _robust_rescale(bmp4), "Greens"),
            ("FOXF1/BMP4", composite, None),
        ]
        for ax, (title, img, cmap) in zip(ax_row, panels):
            if cmap is None:
                ax.imshow(img)
            else:
                ax.imshow(img, cmap=cmap)
            ax.set_title(f"{row.position_label} | z{z_index + 1} | {title}")
            ax.axis("off")

    plt.show()


plot_selected_review_plane_montage()